# Project 2: Similarity Matrix and Language Clustering
Submitted on November 10, 2025

NLP1000 S17
Group 12: 
| Member Name      | ID Number      |
| ------------- | ------------- |
| Alcantara, Van Asher | 12340898 |
| Aragon, Enrique | 12227811 |
| Clavano, Angelica (Jack) | 12206245 |
| Lozada, Job | 12307246 |

## Table of Contents
0. Libraries
1. Data Sourcing & Pre-processing
2. Similarity Matrix
3. Dendrogram
4. Interpretation of the Matrix & Dendrogram
5. Conclusion and Insights
6. Declaration of AI Usage

---

## Running this Notebook:
- Make sure you have at least Python 3.12.0 or 3.14.0 installed.
- To install the requirements, run `pip install -r requirements.txt` It has been updated for usage with `project2.ipynb`

---

## Folder Structure:
```
/nlp1000 --> root
└── /data
    └── lang.txt files --> raw webscraped data
    └── /cleaned
        └── (sentences)-lang-cleaned.txt --> cleaned sentence files in txt format
        └── lang-cleaned.txt --> cleaned verse files in txt format
└── /parallel-corpora --> parallel corpora (aligned) but in separate xlsx files
└── .gitignore
└── main.ipynb --> PROJECT 1 SUBMISSION containing documentation and source code
└── project2.ipynb --> PROJECT 2 SUBMISSION
└── removed.ipynb --> experimental code not part of the main submission
└── cleaned_sentences.ipynb --> cleaned sentence files in xlsx format - from data/cleaned/(sentences)-(lang)-cleaned.txt files
└── README.md --> contains the same information as this markdown block.
└── cleaned_verses.xlsx --> cleaned verse files in xlsx format - created when running the notebook from data/cleaned/(lang)-cleaned.txt
└── parallel_corpora.xlsx --> parallel corpora base file (unaligned) - created when running the notebook 
└── parallel_corpora_all.xlsx --> parallel corpora (aligned)
└── ai_declaration_project1.pdf --> AI declaration
└── ai_declaration_project2.pdf --> AI DECLARATION FOR PROJECT 2
└── steps.xlsx --> steps for regex present in divide_into_verses() and divide_into_sentences()
```

## **0. Libraries**

The following Python libraries/methods were used for this project. It is necessary to install the ones with attached comments through the CLI. 

**nltk** - used to tokenize the text, as well as to generate trigrams and their respective frequency counts

**nltk - TweetTokenizer** - specific tokenizer needed for the text

**pathlib - Path** - used to manipulate the current directory of the notebook for correct file pathfinding

**pandas** - standard data science library used for the creation of the dendrogram

**scipy - dendrogram, linkage** - used to perform hierarchical clustering and create the dendrogram

**matplotlib.pyplot** - used to display the dendrogram\


In [1]:
import nltk                 # pip install nltk
from pathlib import Path    # pip install pathlib
import pandas as pd         # pip install pandas
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

from nltk.tokenize import TweetTokenizer

The code below is just to create a list for iterating through our language files. If someone wishes to add or remove languages in their own recreation of this project, they can edit this list.

In [2]:
languages = ["spanish", "tagalog", "english", "hiligaynon", "bikol", "waray", "ilocano", "cebuano", "kapampangan", "pangasinense", "yakan", "ivatan", "tausug", "yami", "tuwali_ifugao", "masbateno"]

The following code block is commented out as this only needs to be run once. Once this has been run once, or if you already have these libraries installed, **DO NOT** run this code block again, as it will cause issues. Comment it out or run only the code blocks below it instead.

In [3]:
#needed for nltk, you can just run this once
'''
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('popular')'''

"\nnltk.download('punkt')\nnltk.download('punkt_tab')\nnltk.download('popular')"

## **1. Data Sources & Pre-processing**

### 1.1 Data Sourcing

For this project, we opted to use the cleaned-up text from the first project as our corpora, specifically the version split by sentences. Not only is it already preprocessed data, but it also reaches the requirement of 50,000 words per language corpora. Additionally, they are all translations of the same Bible books (Matthew, Mark, Luke, and John) from the Gospel of the New Testament. This will ensure that the generated trigrams and similarity matrix will properly reflect the relationships between each language.


### 1.2 Generating Trigrams with NLTK

For this project, we decided to generate **character trigrams** as our feature set to create the similarity matrix later on. Initially, we considered using each corpora's most common words as the primary criteria used in creating our similarity matrix. However, we decided against it as even minor spelling differences between functionally similar words would result in a mismatch. Character trigrams, on the other hand, seemed to be more indicative of repeating patterns or similarity between languages.

An important note is that we have intentionally replaced the whitespace characters that may show up in the trigrams with _ for visibility purposes. Since all instances of whitespace in the trigrams are replaced, this does not affect the resulting similarity matrix.

In [ ]:
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

data_folder = Path("corpora")
output_folder = Path("all_trigrams")

for lang in languages:
    # define source and output
    source_file_path = data_folder / f"{lang}-corpora.txt"
    output_file_path = output_folder/f"{lang}-all-trigrams.txt"

    # if source doesn't exist skip
    if not source_file_path.exists() or source_file_path.stat().st_size == 0:
            print(f"Skipped: {source_file_path} (file does not exist or is empty)")
            # continue
    else:       
        # rewrite all the txts if it exists
        if output_file_path.exists():
            print(f"unlinking/deleting old files")
            output_file_path.unlink()

        # read input
        with source_file_path.open("r", errors="ignore", encoding="utf-8") as f:
            raw_sentences = f.read()

        # DO WHATEVER YOU WANT HERE
        # tokens = word_tokenize(raw_sentences) # split into words - DON'T DELETE THIS THANKS
        characters = [c for c in raw_sentences]

        # if you want to do it in words replace "characters" with "tokens"
        # unigrams = list(ngrams(characters, 1))
        # bigrams = list(ngrams(characters, 2))
        trigrams = list(ngrams(characters, 3))

        # save them to the file
        with output_file_path.open("w", encoding="utf-8") as f:
            f.write("Trigrams\n")
            for list_tuple in trigrams:
                # replace space with _ for readability and so the tab actually works in top 250
                cleaned = ( '_' if (isinstance(ch, str) and ch.isspace()) else str(ch) for ch in list_tuple )
                f.write("".join(cleaned) + "\n")
                # f.write(" ".join(map(str, list_tuple)) + "\n") - DONT DELETE THIS THANKS

        print(f"Cleaned and saved: {output_file_path}")


Cleaned and saved: all_trigrams\spanish-all-trigrams.txt
Cleaned and saved: all_trigrams\tagalog-all-trigrams.txt
Cleaned and saved: all_trigrams\english-all-trigrams.txt
Cleaned and saved: all_trigrams\hiligaynon-all-trigrams.txt
Cleaned and saved: all_trigrams\bikol-all-trigrams.txt
Cleaned and saved: all_trigrams\waray-all-trigrams.txt
Cleaned and saved: all_trigrams\ilocano-all-trigrams.txt
Cleaned and saved: all_trigrams\cebuano-all-trigrams.txt
Cleaned and saved: all_trigrams\kapampangan-all-trigrams.txt
Cleaned and saved: all_trigrams\pangasinense-all-trigrams.txt
Cleaned and saved: all_trigrams\yakan-all-trigrams.txt
Cleaned and saved: all_trigrams\ivatan-all-trigrams.txt
Cleaned and saved: all_trigrams\tausug-all-trigrams.txt
Cleaned and saved: all_trigrams\yami-all-trigrams.txt
Cleaned and saved: all_trigrams\tuwali_ifugao-all-trigrams.txt
Cleaned and saved: all_trigrams\masbateno-all-trigrams.txt


After generating the character trigrams found in each language's corpora, along with their respective frequencies, we used NLTK's most_common() method to generate files containing only the top 250 most common character trigrams per language. While we initially considered 1000 trigrams or more to account for the sheer size each corpora, we realized that the type of similarity we planned to use for the similarity matrix, the Dice coefficient, did not account for the frequency of each trigram. If we treated the most common character trigram with the same level of relevance as the 1000th most common character trigram, then the resulting similarity matrix may misrepresent how similar the languages actually are.

In [10]:
from collections import Counter
data_folder = Path("all_trigrams")
output_folder = Path("t250-trigrams")

for lang in languages:
    # define source and output
    source_file_path = data_folder / f"{lang}-all-trigrams.txt"
    output_file_path = output_folder/f"{lang}-t250.txt"

    # if source doesn't exist skip
    if not source_file_path.exists() or source_file_path.stat().st_size == 0:
            print(f"Skipped: {source_file_path} (file does not exist or is empty)")
            # continue
    else:       
        # rewrite all the txts if it exists
        if output_file_path.exists():
            print(f"unlinking/deleting old files")
            output_file_path.unlink()

        # read input
        with source_file_path.open("r", errors="ignore", encoding="utf-8") as f:
            next(f, None) # skip the first line bc it's just a title
            trigrams = [line.strip() for line in f if line.strip()]

        freq_counter = Counter(trigrams)
        top_250_trigrams = freq_counter.most_common(250)

        # save them to the file
        with output_file_path.open("w", encoding="utf-8") as f:
            f.write(f"{lang}top250\n") # f.write("Top 250\tFrequency\n")
            for trigram, count in top_250_trigrams:
                 f.write(f"{trigram} {count}\n") # f.write(f"{trigram}\t\t{count}\n")

        print(f"Cleaned and saved: {output_file_path}")
        

unlinking/deleting old files
Cleaned and saved: t250-trigrams\spanish-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\tagalog-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\english-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\hiligaynon-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\bikol-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\waray-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\ilocano-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\cebuano-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\kapampangan-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\pangasinense-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\yakan-t250.txt
unlinking/deleting old files
Cleaned and saved: t250-trigrams\ivatan-t250.txt
unlinking/deleting old files
Cleaned and saved:

After generating the top 250 character trigrams of each language, we manually imported each txt file into Google Sheets as a CSV in order to line them up with each other. The CSV created from this is what we will use as a basis for our similarity matrix.

The top 250 character trigrams per language can be found [here](https://docs.google.com/spreadsheets/d/13XduT75m-G1xUqucKRqv8CBZWuz_OuDsENqoi2Qua_w/edit?usp=sharing).

<span style="background-color:red"> EVERYTHING AFTER HERE WE NEED TO CHANGE I THINK </span>

## 2. Similarity and Clustering / Dendrogram

## 3. Creating the Dendrogram

## 4. Interpretation of the Matrix & Dendrogram

## 5. Conclusion & Insights


## 6. Declaration of AI Usage

NLP1000 S17 Project 2
**Group 12: AI Declaration**
| Name | ID number | 
| --- | --- |
| Van Asher Alcantara | 12340898 | 
| Enrique Miguel M. Aragon | 12227811 | 
| Angelica Therese Clavano (Jack) | 12206245 | 
| Job Lozada | 12307246 | 

**Declaration of AI usage: Van Asher Alcantara**


I used ChatGPT for paraphrasing and grammar checking some parts of the writeup for this project.
**Extent of use:** Minimal – used only for grammar/spell check/definition.
**Prompts and Answers:**
Prompt: define shared vocabulary retention
ChatGPT:
**Shared vocabulary retention** refers to the ability of a group of people (such as members of a community, team, or culture) to **maintain and consistently use a common set of words, phrases, or terminologies** over time.
In other words, it describes **how well a group preserves its common linguistic repertoire**—the shared meanings and terms that allow members to communicate effectively and understand each other.
Here’s how it breaks down:
* **“Shared vocabulary”** means the set of words or expressions known and used by members of a particular group.
* **“Retention”** means keeping or maintaining something over time.
So, **shared vocabulary retention** can be defined as:
*The sustained use and preservation of a common set of words or terms within a community or group, ensuring continued mutual understanding and communication coherence over time.*
**Explanation**:
AI is very helpful when it comes to paraphrasing and finding definitions of uncommon words to possibly replace them with simpler terms. Additionally, it can also give suggestions on how to write after given a context and setting.

---

**Declaration of AI usage: Enrique Aragon**


I used Deepseek to verify my knowledge and understanding of how to create a dendrogram after finishing the similarity matrix, since the last language tree I created was with Weka, rather than Python.
**Extent of use:** Minimal – used for knowledge verification
**Prompts and Answers:**
Prompt: The similarity matrix is already generated using dice coefficients. Do I still convert it into a distance matrix to make a dendrogram?
Deepseek:
**Yes, for Dice coefficient you still need distance conversion:**
Dice coefficient = similarity (0 = no overlap, 1 = identical)
Use:
distance = 1 - Dice_coefficient
So:
* Dice = 1 → distance = 0 (identical languages)
* Dice = 0 → distance = 1 (completely different)
**Alternative for Dice**: Some use 1 - Dice or sqrt(1 - Dice) as distance metric. 1 - Dice is most common and appropriate for language relationships.

**Explanation**:
I believe that AI was helpful in learning as I could use it as a source for verifying my understanding of the process of making a similarity matrix and a dendrogram. I did not use it as my only source of information, and cross-checked as well with other standard online resources to ensure I was doing it properly.

---

**Declaration of AI usage: Jack Clavano**


I used __ChatGPT 5__ for __(idea generation, grammar checking, paraphrasing, summarizing references, formatting help, etc.). __
**Extent of use**: (choose one)
* Minimal – used only for grammar/spell check.
* Moderate – used for inspiration, then reworded significantly
**Prompts and Answers:**
PROMPT: what library from python can i use to get the frequency count given a file of character trigrams?
ChatGPT 5: "You can use Python’s collections module — specifically, the Counter class — to get frequency counts from a file of character trigrams. Here’s why it’s the best choice:
Part of the standard library — no installation required.
Optimized for counting hashable items like strings or tuples.
Simple API with methods like .most_common() to get the top-N frequent items."

PROMPT: does counter library account for spaces?
ChatGPT 5:  "Yes — collections.Counter does account for spaces exactly as they appear in your data.
It treats each line (or string) as a distinct key, meaning that spaces, tabs, and all other characters are preserved and counted literally"

**Explanation**: In 2–3 sentences, reflect on how the use of AI contributed to your learning. If no AI was used, reflect on why you chose not to.
Usage of AI contributes to learning because it is able to give you quick responses to searches that would otherwise take way longer. It can lead you to the right or similar answers to your questions.

---

**Declaration of AI usage: Job Lozada**


I used Google Gemini to help find academic sources to link and differentiate Ilocano and Pangasinese
**Extent of use:** (choose one)
* Minimal – aid in finding sources
**Prompts and Answers:**
**Prompt: **Ilocano and Pangasinese are two languages that appear to be closely related despite their geographical differences. Are there academic papers that look into their history and relationship?
**Answer:**
For more academic resources on the relationship between the Ilocano and Pangasinense peoples, the following published articles, books, and studies can be consulted:
Scholarly Articles and Papers

"Pangasinan—An Endangered Language? Retrospect and Prospect" by N. G. Fernandez and M. L. Rosario (published in Philippine Studies, 2007; also available on ResearchGate and Archīum Ateneo).
Focus: This widely-cited paper is a key resource on the linguistic dynamics in Pangasinan. It specifically discusses how Ilocano migration, interethnic marriage, and cultural prestige have led to a significant Ilocano-speaking population in the province, which in turn influences the vitality of the indigenous Pangasinan language.
"The Ilocano Diaspora to Cagayan: A Historical Study of Migration" by A. R. Flores Jr. (published by the Philippine Social Science Council, 2009).
Focus: While primarily about migration to Cagayan, this research provides a general framework and context for the Ilocano historical propensity to migrate due to resource competition in their homeland, a pattern that also applies to their movement into Pangasinan.
"Reclaiming Identity toward Decolonisation: Pangasinan Studies in Theory and Praxis" by C. C. Quitalig (published on ResearchGate, 2017).
Focus: This work addresses the cultural and historical marginalization of Pangasinan culture and language, noting how Ilocano migrations have impacted the linguistic landscape and the need for the revitalization of Pangasinan heritage and studies.
"Exploring the Pangasinan-Cordillera Connection: The Pangasinenses and the Ibalois" by C. A. Pawilen and A. P. Arellano (published in The Cordillera Review, 2022).
Focus: This paper examines the historical and pre-colonial connections between the Pangasinenses and their neighboring groups, which helps provide context for their interactions with Ilocanos, particularly concerning trade and settlement patterns.

Books and Reference Materials

"The Ethnohistory of Northern Luzon" by Felix Keesing (Stanford University Press, 1962).
Focus: A foundational anthropological and historical text that details the various ethnic groups of Northern Luzon, including the Ilocano and Pangasinense, their settlement patterns, and early interactions.
"Pangasinan Reference Grammar" and "Pangasinan Dictionary" by Thomas N. Headland (published as part of the PALI Language Texts series by the University of Hawai'i Press).
Focus: These linguistic resources offer in-depth documentation of the Pangasinan language, useful for academic comparisons with Ilocano to understand their distinct origins and any areas of lexical borrowing.

General Academic Repositories

JSTOR, ResearchGate, Archīum Ateneo, and the PSSC (Philippine Social Science Council) archives are excellent places to search for further studies, using terms like "Ilocano migration Pangasinan," "Pangasinan language shift," or "Ilocano-Pangasinan intermarriage".
**Explanation**:
The AI helped me learn more about regional languages by exposing me to academic research into them, particularly the role of immigration and social influence in causing languages to change and become more similar.

## References
- Anderson, Victoria & Anderson, James. 2007. Pangasinan – An Endangered Language:
Retrospect and Prospect. Philippine Studies, 55,
http://www2.hawaii.edu/~vanderso/Pangasinan.pdf (accessed November 9, 2025).

- Molony, C. H. (1972). A chronology of the Sulu region. Borneo Research Bulletin, 4(1), 24–25.

- Global Recordings Network. (n.d.). Language prayer profile: Waray. Retrieved November 9, 2025, from https://globalrecordings.net/en/profile-waray

- Anderson, V. B., & Anderson, J. N. (2007). Pangasinan—An Endangered Language? Retrospect and Prospect. Philippine Studies, 55(1), 116–144. http://www.jstor.org/stable/42633901

- Besonia, Bon Eric. (2022). International Journal of Multidisciplinary Approach and Studies The Death of Hiligaynon Terminologies in the Coastal Community in Northern Iloilo, Philippines. 

- ARCE-DAET A. (2024, November). LANGUAGE DISTRIBUTION IN THE PROVINCE OF APAYAO. GreenField Advanced Research Publishing House. https://garph.co.uk/IJARMSS/Nov2024/G-3093.pdf

- Archive, E. L. (n.d.). Yami documentation. | Endangered Languages Archive. Retrieved November 8, 2025, from https://www.elararchive.org/dk0110/

- Cebuano. (n.d.). Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/ceb/

- Defense Language Institute Foreign Language Center. (2009, September). Yakan Cultural Orientation - Technology Integration Division. FAMiliarization. https://fieldsupport.dliflc.edu/products/yakan/yn_co/Yakan.pdf

- DeFraga, C. (n.d.). Language Group Specific Informational Reports - Cebuano. RITELL - Rhode Island Teachers of English Language Learners News. https://www.ritell.org/Resources/Documents/language%20project/Cebuano.pdf

- English language. (2025, October 31). Encyclopedia Britannica. Retrieved November 8, 2025, from https://www.britannica.com/topic/English-language

- Ethnologue. (n.d.). Ivatan. Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/ivv/

- Ethnologue. (n.d.). Yakan. Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/yka/

- Ethnologue. (n.d.). Yami. Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/tao/

- EverythingExplained. (n.d.). Ivatan language explained. Everything Explained Today. Retrieved November 8, 2025, from https://everything.explained.today/Ivatan_language/

- Forman, M. L. (1971). KAPAMPANGAN GRAMMAR NOTES. University of Hawaii Press. https://scholarspace.manoa.hawaii.edu/server/api/core/bitstreams/643e6458-31d8-4e86-99d7-c001d804b510/content

- Ilocano. (n.d.). Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/ilo/

- Kapampangan. (n.d.). Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/pam/

- Mabuan, R., Dita, S., & Tanangkingsing, M. (2022, October). Word formation processes in Masbatenyo. ACL Anthology. https://aclanthology.org/2022.paclic-1.32/

- Madeja, J. B., Lara, M. L., & Abenis, A. B. (2017, January). Waray Morphology. Google Scholar. https://scholar.google.com/citations?view_op=view_citation&hl=en&user=NNc6YiQAAAAJ&citation_for_view=NNc6YiQAAAAJ:UeHWp8X0CEIC

- Malabonga, V. (2009, January). Heritage Voices: Language-Tagalog. ResearchGate. https://www.researchgate.net/publication/319401768_Heritage_Voices_Language-Tagalog

- Omniglot. (n.d.). Ivatan language and alphabet. Omniglot - the encyclopedia of writing systems and languages. Retrieved November 8, 2025, from https://www.omniglot.com/writing/ivatan.htm

- (n.d.). Redirecting you to University of Hawaii... https://www2.hawaii.edu/~vanderso/Pangasinan.pdf

- Reid, L. (2005, January 1). Tagalog and Philippine languages. Academia.edu - Find Research Papers, Topics, Researchers. https://www.academia.edu/23925018/Tagalog_and_Philippine_Languages

- SinaunangPanahon. (2025, April 24). Kapampangan language of the Philippines. SINAUNANGPANAHON. Retrieved November 8, 2025, from https://sinaunangpanahon.com/kapampangan-language-of-the-philippines/

- Spanish language. (2025, October 31). Encyclopedia Britannica. Retrieved November 8, 2025, from https://www.britannica.com/topic/Spanish-language

- Tahil, S., & Alibasa, J. T. (2023, November). Preserving and Nurturing Tausug Language: The Bahasa Sug Mobile Learning Application Tool for Enhancing Mother Tongue Development for Toddlers. ResearchGate. https://www.researchgate.net/publication/376075255_Preserving_and_Nurturing_Tausug_Language_The_Bahasa_Sug_Mobile_Learning_Application_Tool_for_Enhancing_Mother_Tongue_Development_for_Toddlers

- Taleon, K. A. (2020, September). A Phonological Sketch of Tuwali Ifugao. ResearchGate. https://www.researchgate.net/publication/344170392_A_Phonological_Sketch_of_Tuwali_Ifugao

- Tobian. (2024, November 17). Spanish vs English language: Key differences & history. Tobian Language School. Retrieved November 8, 2025, from https://tobian-languageschool.com/spanish-vs-english-language-key-differences-history-and-fascinating-comparisons/

- What are the origins of the English language? | Merriam-Webster. (n.d.). Merriam-Webster: America's Most Trusted Dictionary. Retrieved November 8, 2025, from https://www.merriam-webster.com/help/faq-history

- Wyland, J. (2025, August 19). The history of Spanish language: From Latin roots to a global reach. The University of Texas Permian Basin | UTPB. https://online.utpb.edu/about-us/articles/spanish/the-history-of-spanish-language-from-latin-roots-to-a-global-reach/

- Yami. (n.d.). Ethnologue | Languages of the world. Retrieved November 8, 2025, from https://www.ethnologue.com/language/tao/

- Zorc, R. D. (2001). Hiligaynon. Facts About the World's Languages: An Encyclopedia of the World's Major Languages, Past and Present. https://ia800207.us.archive.org/9/items/rosettaproject_hil_detail-1/rosettaproject_hil_detail-1.pdf

- Zorc, R. D., Lobel, J. W., & Hall, W. (n.d.). The History of the Philippine Languages. Zorc.net. https://zorc.net/publications/142a-submitted=Philippines_Historical_Chapter[Zorc-Lobel-Hall].pdf